In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10


@dataclass(frozen=True)
class Paths:
    main_workbook: Path = Path("data") / "data.xlsx"
    output_dir: Path = Path("outputs") / "table_2_figure_s6"
    positive_sheet: str = "Positive"
    n_pca_components: int = 5


CLASS_ORDER = ["Lamp oil", "White spirit", "Diesel", "Gasoline"]

LEVEL2_EXTERNAL_HOLDOUT_ROOTS = {
    "T1", "T2", "T4", "T5", "T7", "T10", "T11", "T13", "T14",
    "T6", "T9", "T12", "T15",
    "Te3", "Te6",
    "W1", "W2", "W3",
    "L1", "L3", "L7",
}

DIESEL_ROOTS = {
    "SH3", "SH4", "SH7", "SH8", "SH11", "SH12", "SH15", "SH16", "SH19", "SH20",
    "T3", "T6", "T9", "T12", "T15",
    "Te3", "Te6", "Te9", "Te12", "Te15",
}

GASOLINE_95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17",
    "T1", "T4", "T7", "T10", "T13",
    "Te1", "Te4", "Te7", "Te10", "Te13",
}

GASOLINE_98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18",
    "T2", "T5", "T8", "T11", "T14",
    "Te2", "Te5", "Te8", "Te11", "Te14",
}

GASOLINE_ROOTS = GASOLINE_95_ROOTS.union(GASOLINE_98_ROOTS)


def root_code(sample_id: str) -> str:
    return str(sample_id).strip().split("-", 1)[0]


def class_label_from_root(root: str) -> str:
    if root.startswith("B"):
        raise ValueError("Brandspiritus must be excluded from this four-class workflow.")
    if root in DIESEL_ROOTS:
        return "Diesel"
    if root in GASOLINE_ROOTS:
        return "Gasoline"
    if root.startswith("L"):
        return "Lamp oil"
    if root.startswith("W"):
        return "White spirit"
    raise ValueError(f"Unknown root code for class mapping: {root}")


def classes_present(labels: np.ndarray) -> List[str]:
    present = set(np.unique(labels).tolist())
    return [class_name for class_name in CLASS_ORDER if class_name in present]


def class_key(class_name: str) -> str:
    return class_name.replace(" ", "_")


def sorted_numeric_columns(df: pd.DataFrame) -> List[str]:
    numeric_columns: List[Tuple[float, str]] = []
    for column in df.columns:
        try:
            numeric_columns.append((float(column), column))
        except (TypeError, ValueError):
            continue

    if not numeric_columns:
        raise ValueError("No numeric spectral columns were found in the Positive worksheet.")

    numeric_columns.sort(key=lambda item: item[0])
    return [column for _, column in numeric_columns]


def load_dev_and_external_data(
    paths: Paths,
) -> Tuple[np.ndarray, pd.DataFrame, np.ndarray, pd.DataFrame]:
    df = pd.read_excel(paths.main_workbook, sheet_name=paths.positive_sheet, index_col=0)
    df.index = df.index.to_series().astype(str)

    roots = df.index.to_series().map(root_code).astype(str)
    keep_mask = ~roots.str.startswith("B")
    df = df.loc[keep_mask].copy()
    roots = roots.loc[keep_mask].copy()

    spectral_columns = sorted_numeric_columns(df)
    X = df[spectral_columns].to_numpy(dtype=float)
    labels = np.array([class_label_from_root(root) for root in roots], dtype=object)
    metadata = pd.DataFrame(
        {"root": roots.to_numpy(), "class_label": labels},
        index=df.index,
    )

    external_mask = roots.isin(LEVEL2_EXTERNAL_HOLDOUT_ROOTS).to_numpy()
    development_mask = ~external_mask

    if not np.any(development_mask):
        raise ValueError("The development set is empty.")
    if not np.any(external_mask):
        raise ValueError("The external test set is empty.")

    X_dev = X[development_mask]
    meta_dev = metadata.loc[development_mask].copy()
    X_external = X[external_mask]
    meta_external = metadata.loc[external_mask].copy()
    return X_dev, meta_dev, X_external, meta_external


def preprocess_snv(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("SNV preprocessing expects a two-dimensional array.")

    means = X.mean(axis=1, keepdims=True)
    standard_deviations = X.std(axis=1, ddof=1, keepdims=True)
    standard_deviations[standard_deviations == 0.0] = 1.0
    return (X - means) / standard_deviations


def fit_pca_lda(
    X_dev: np.ndarray,
    y_dev: np.ndarray,
    n_pca_components: int,
) -> Tuple[PCA, LinearDiscriminantAnalysis]:
    classes = np.unique(y_dev)
    missing_classes = set(CLASS_ORDER).difference(classes)
    if missing_classes:
        raise ValueError(f"Development data are missing classes: {sorted(missing_classes)}")

    max_components = min(n_pca_components, X_dev.shape[1], X_dev.shape[0] - 1)
    if max_components < 2:
        raise ValueError("At least two PCA components are required for the LDA plot.")

    pca = PCA(n_components=max_components)
    X_dev_pca = pca.fit_transform(X_dev)

    equal_priors = np.full(classes.shape[0], 1.0 / classes.shape[0])
    lda = LinearDiscriminantAnalysis(priors=equal_priors)
    lda.fit(X_dev_pca, y_dev)
    return pca, lda


def lda_axis_label(lda: LinearDiscriminantAnalysis, index: int) -> str:
    ratios = getattr(lda, "explained_variance_ratio_", None)
    if ratios is None:
        return f"LD{index + 1}"

    ratios = np.asarray(ratios, dtype=float)
    total = ratios.sum()
    if ratios.ndim != 1 or len(ratios) <= index or not np.isfinite(total) or total <= 0:
        return f"LD{index + 1}"

    normalized_ratios = ratios / total
    return f"LD{index + 1} ({normalized_ratios[index] * 100:.1f}% discriminant ability)"


def style_ld_axes(ax: plt.Axes) -> None:
    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_top()
    ax.yaxis.set_label_position("right")
    ax.yaxis.tick_right()
    ax.yaxis.label.set_rotation(270)
    ax.yaxis.label.set_verticalalignment("center")
    ax.tick_params(
        axis="x",
        which="both",
        top=True,
        labeltop=True,
        bottom=False,
        labelbottom=False,
        direction="in",
        length=5,
        labelsize=11,
    )
    ax.tick_params(
        axis="y",
        which="both",
        right=True,
        labelright=True,
        left=False,
        labelleft=False,
        direction="in",
        length=5,
        labelsize=11,
    )
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["top"].set_visible(True)
    ax.spines["top"].set_linewidth(1.1)
    ax.spines["top"].set_color("black")
    ax.spines["right"].set_visible(True)
    ax.spines["right"].set_linewidth(1.1)
    ax.spines["right"].set_color("black")


def save_external_ld_plot(
    lda_scores: np.ndarray,
    labels: np.ndarray,
    lda: LinearDiscriminantAnalysis,
    output_dir: Path,
) -> Path:
    if lda_scores.ndim != 2 or lda_scores.shape[1] < 2:
        raise ValueError("The fitted LDA model did not produce both LD1 and LD2.")

    fig, ax = plt.subplots(figsize=(8.2, 6.4))
    ax.set_facecolor("#f5f5f5")
    color_map = plt.get_cmap("tab10")

    for index, class_name in enumerate(classes_present(labels)):
        class_mask = labels == class_name
        ax.scatter(
            lda_scores[class_mask, 0],
            lda_scores[class_mask, 1],
            s=30,
            alpha=0.85,
            edgecolors="none",
            color=color_map(index),
            label=class_name,
        )

    ax.set_xlabel(lda_axis_label(lda, 0), fontsize=14, family="serif")
    ax.set_ylabel(lda_axis_label(lda, 1), fontsize=14, family="serif")
    style_ld_axes(ax)
    ax.grid(True, color="#c8c8c8", linewidth=0.6, alpha=0.8)

    legend = ax.legend(loc="best", frameon=True)
    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_edgecolor("black")

    fig.tight_layout()
    output_path = output_dir / "SNV_LDA_LD1_vs_LD2_external_test.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return output_path


def build_external_prediction_table(
    lda: LinearDiscriminantAnalysis,
    X_external_pca: np.ndarray,
    y_external: np.ndarray,
) -> pd.DataFrame:
    predicted_labels = lda.predict(X_external_pca)
    decision_scores = np.asarray(lda.decision_function(X_external_pca), dtype=float)
    classes = np.asarray(lda.classes_, dtype=object)

    if decision_scores.ndim == 1:
        decision_scores = decision_scores.reshape(-1, 1)
    if decision_scores.shape[1] != len(classes):
        raise ValueError("Unexpected number of LDA decision-score columns.")

    table = pd.DataFrame(
        {"true_label": y_external, "predicted_label": predicted_labels}
    )
    for index, class_name in enumerate(classes):
        table[f"lda_decision_score_{class_key(class_name)}"] = decision_scores[:, index]
    return table


def build_confusion_matrix_values(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    matrix = pd.DataFrame("", index=CLASS_ORDER, columns=CLASS_ORDER, dtype=object)

    for true_class in CLASS_ORDER:
        true_mask = predictions["true_label"].to_numpy(dtype=object) == true_class
        total = int(np.count_nonzero(true_mask))
        if total:
            correct = int(np.count_nonzero(
                predictions.loc[true_mask, "predicted_label"].to_numpy(dtype=object) == true_class
            ))
            matrix.loc[true_class, true_class] = f"{correct}/{total}"

    for assigned_class in CLASS_ORDER:
        assigned_column = f"lda_decision_score_{class_key(assigned_class)}"
        assigned_mask = (
            predictions["predicted_label"].to_numpy(dtype=object) == assigned_class
        )
        if assigned_column not in predictions.columns or not np.any(assigned_mask):
            continue

        assigned_scores = predictions.loc[assigned_mask, assigned_column].to_numpy(dtype=float)
        for competing_class in CLASS_ORDER:
            if competing_class == assigned_class:
                continue
            competing_column = f"lda_decision_score_{class_key(competing_class)}"
            competing_scores = predictions.loc[assigned_mask, competing_column].to_numpy(dtype=float)
            log10_lr = (assigned_scores - competing_scores) / np.log(10.0)
            minimum_log10_lr = float(np.min(log10_lr))
            matrix.loc[competing_class, assigned_class] = f"{minimum_log10_lr:.2f}"

    return matrix


def save_confusion_matrix_excel(
    matrix: pd.DataFrame,
    output_dir: Path,
) -> Path:
    output_path = output_dir / "SNV_PCA_LDA_confusion_matrix_external_test.xlsx"

    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        workbook = writer.book
        worksheet = workbook.add_worksheet("External test")
        writer.sheets["External test"] = worksheet

        top_format = workbook.add_format({
            "bold": True,
            "align": "center",
            "valign": "vcenter",
            "border": 1,
            "font_name": "Times New Roman",
            "font_size": 12,
        })
        header_format = workbook.add_format({
            "align": "center",
            "valign": "vcenter",
            "border": 1,
            "font_name": "Times New Roman",
            "font_size": 12,
        })
        row_header_format = workbook.add_format({
            "align": "left",
            "valign": "vcenter",
            "border": 1,
            "font_name": "Times New Roman",
            "font_size": 12,
        })
        diagonal_format = workbook.add_format({
            "align": "center",
            "valign": "vcenter",
            "border": 1,
            "bg_color": "#92D050",
            "font_name": "Times New Roman",
            "font_size": 12,
        })
        body_format = workbook.add_format({
            "align": "center",
            "valign": "vcenter",
            "border": 1,
            "font_name": "Times New Roman",
            "font_size": 12,
        })
        start_row = 0
        worksheet.merge_range(
            start_row,
            1,
            start_row,
            len(CLASS_ORDER),
            "Assigned →",
            top_format,
        )
        worksheet.write(start_row + 1, 0, "True ↓", top_format)

        for column, class_name in enumerate(CLASS_ORDER, start=1):
            worksheet.write(start_row + 1, column, class_name, header_format)

        for row, true_class in enumerate(CLASS_ORDER, start=start_row + 2):
            worksheet.write(row, 0, true_class, row_header_format)
            for column, assigned_class in enumerate(CLASS_ORDER, start=1):
                value = matrix.loc[true_class, assigned_class]
                if true_class == assigned_class:
                    cell_format = diagonal_format
                else:
                    cell_format = body_format
                worksheet.write(row, column, value, cell_format)

        worksheet.set_column(0, 0, 18)
        worksheet.set_column(1, len(CLASS_ORDER), 16)
        for row in range(start_row, start_row + 2 + len(CLASS_ORDER)):
            worksheet.set_row(row, 22)

    return output_path


def main() -> None:
    paths = Paths()
    paths.output_dir.mkdir(parents=True, exist_ok=True)

    X_dev_raw, meta_dev, X_external_raw, meta_external = load_dev_and_external_data(paths)
    X_dev_snv = preprocess_snv(X_dev_raw)
    X_external_snv = preprocess_snv(X_external_raw)

    y_dev = meta_dev["class_label"].to_numpy()
    y_external = meta_external["class_label"].to_numpy()
    pca, lda = fit_pca_lda(X_dev_snv, y_dev, paths.n_pca_components)

    X_external_pca = pca.transform(X_external_snv)
    external_lda_scores = np.asarray(lda.transform(X_external_pca), dtype=float)
    figure_path = save_external_ld_plot(
        external_lda_scores,
        y_external,
        lda,
        paths.output_dir,
    )

    predictions = build_external_prediction_table(lda, X_external_pca, y_external)
    confusion_matrix = build_confusion_matrix_values(predictions)
    table_path = save_confusion_matrix_excel(
        confusion_matrix,
        paths.output_dir,
    )

    print(f"Saved Figure S6: {figure_path}")
    print(f"Saved Table 2: {table_path}")


if __name__ == "__main__":
    main()
